In [ ]:
import scipy.io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler  # 添加缺失的导入
from sklearn.ensemble import RandomForestRegressor  # 添加缺失的导入
from sklearn.multioutput import MultiOutputRegressor  # 添加缺失的导入

# 设置 Times New Roman 字体
plt.rcParams['font.sans-serif'] = ['Times New Roman']  # 使用 Times New Roman 字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题

# 1. 加载数据
print("正在加载数据...")
df_augmented = pd.read_excel('../data/CTGAN_data_multi.xlsx')
df_original = pd.read_excel('../data/new_data_x_names.xlsx')
print(f"增强数据集大小: {df_augmented.shape}")
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']

# 3. 分离特征和目标
# 从增强数据中分离
X_augmented = df_augmented.drop(target_vars, axis=1)
y_augmented = df_augmented[target_vars]

# 从原始数据中分离（用于最终评估）
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为测试集
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

# 5. 数据标准化
scaler_X = StandardScaler()
X_augmented_scaled = scaler_X.fit_transform(X_augmented)
X_test_scaled = scaler_X.transform(X_test_orig)  # 使用相同的缩放器

# 6. 创建并训练模型
print("\n开始训练随机森林模型...")
model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    n_jobs=-1,
    random_state=42
))

# 使用增强数据训练
model.fit(X_augmented_scaled, y_augmented)
print("模型训练完成！")

# 7. 在测试集上进行预测
y_pred = model.predict(X_test_scaled)



In [ ]:
# 添加代码到已有的Notebook中
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.metrics import r2_score

# 创建保存图像的目录
save_dir = './figures'
if not os.path.exists(save_dir):
    os.makedirs(save_dir)

# 设置 Times New Roman 字体
plt.rcParams['font.sans-serif'] = ['Times New Roman']
plt.rcParams['axes.unicode_minus'] = False

# 为避免变量混淆，重命名预测结果
predictions = y_pred  # 保存原始预测结果

# 遍历所有目标变量并绘制预测结果与置信区间
for i, var_name in enumerate(target_vars):
    # 获取当前变量的真实值和预测值
    y_true = y_test_orig[var_name].values
    y_pred_var = predictions[:, i]  # 使用不同的变量名
    
    # 计算残差和标准差
    residuals = y_true - y_pred_var
    sigma = np.std(residuals)
    
    # 计算三种置信区间
    conf_intervals = {
        'mu ± sigma': (y_pred_var - sigma, y_pred_var + sigma),
        'mu ± 2sigma': (y_pred_var - 2*sigma, y_pred_var + 2*sigma),
        'mu ± 3sigma': (y_pred_var - 3*sigma, y_pred_var + 3*sigma),
    }
    
    # 绘制可视化图像
    test_indices = np.arange(len(y_true))
    
    plt.figure(figsize=(10, 5), dpi=300)  # 调整dpi以避免过大的文件
    
    # 绘制测试数据
    plt.scatter(test_indices, y_true, color='red', label='Test data', zorder=5)
    
    # 绘制预测均值
    plt.plot(test_indices, y_pred_var, color='blue', label='Prediction, $\\mu$', zorder=4)
    
    # 绘制三种置信区间
    plt.fill_between(test_indices, conf_intervals['mu ± sigma'][0], conf_intervals['mu ± sigma'][1],
                    color='navy', alpha=0.6, label='$\\mu \\pm \\sigma$')
    plt.fill_between(test_indices, conf_intervals['mu ± 2sigma'][0], conf_intervals['mu ± 2sigma'][1],
                    color='deepskyblue', alpha=0.5, label='$\\mu \\pm 2\\sigma$')
    plt.fill_between(test_indices, conf_intervals['mu ± 3sigma'][0], conf_intervals['mu ± 3sigma'][1],
                    color='lightblue', alpha=0.3, label='$\\mu \\pm 3\\sigma$')
    
    # 设置图表标题和标签
    plt.title(f'Test Data with Prediction and Confidence Intervals ({var_name})')
    plt.xlabel('Test Sample')
    plt.ylabel(f'{var_name} Value')
    plt.legend()
    
    # 保存图像
    plt.savefig(f"{save_dir}/Prediction_CI_{var_name}.png", dpi=300, bbox_inches='tight')
    
    # 显示图像
    plt.show()
    
    # 输出评估指标
    r2 = r2_score(y_true, y_pred_var)
    rmse = np.sqrt(np.mean((y_true - y_pred_var)**2))
    print(f"Variable: {var_name}")
    print(f"R² Score: {r2:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Confidence Intervals:")
    print(f"  μ ± σ: {sigma:.4f}")
    print(f"  μ ± 2σ: {2*sigma:.4f}")
    print(f"  μ ± 3σ: {3*sigma:.4f}")
    print("-" * 50)